# 04. Automated Reports & Pipelines in `dyno.py`

When working with macroeconomic models, communicating results through clear, standardized reports is essential.

`dyno.py` provides built-in tools for:
1. **Pipeline Execution (`model.run()`)**: Executing end-to-end DSGE workflows defined by `@run:` directives.
2. **Automated DSGE Reports (`dsge_report()`)**: Generating interactive HTML and Markdown model summaries.
3. **Integration with JupyterLab Dyno Extension (`jupyterlab-dyno`)**: Live interactive side-by-side editing.


In [1]:
import numpy as np
import pandas as pd
from dyno import DynoModel
from dyno.report import dsge_report

pd.set_option('display.precision', 4)


## 1. The `model.run()` Pipeline

`.dyno` files can include declarative `@run:` directives at the bottom of the file. For example, in `models/rbc.dyno`:
```text
@run: check
@run: solve
```

Calling `model.run()` executes these steps sequentially and packages all outputs into a `RunResults` object:


In [2]:
model = DynoModel("models/rbc.dyno")

# Execute the pipeline
results = model.run()

print(f"Results object type: {type(results)}")
print(f"Available fields   : {[f for f in dir(results) if not f.startswith('_')]}")


Results object type: <class 'dyno.report.RunResults'>
Available fields   : ['add_error', 'add_warning', 'bk_check', 'console_display', 'display', 'eigenvalues', 'elapsed', 'errors', 'figure', 'finish', 'jupyter_display', 'mime_bundle_repr', 'model', 'moments', 'output_type', 'residuals', 'simulation', 'solution', 'source_txt', 'warnings']


/tmp/ipykernel_220434/210313483.py:4: UserWarning: DynoModel.run() is experimental and its command metadata format may change.
  results = model.run()


### Accessing Pipeline Outputs
The `RunResults` object gives you convenient programmatic access to each stage of the analysis:


In [3]:
# 1. Model & Solution
print("Solution object:", results.solution)

# 2. Blanchard-Kahn check passed?
print("BK check status :", results.bk_check)

# 3. Equation residuals
print("Max residual    :", np.max(np.abs(results.residuals)))

# 4. Eigenvalues
print("Eigenvalues     :", results.eigenvalues)


Solution object: <dyno.solver.PerturbationSolution object at 0x778ef2f67920>
BK check status : True
Max residual    : 8.881784197001252e-16
Eigenvalues     : [0.         0.         0.         0.         0.         0.
 0.9        0.93984577 1.08020747        inf        inf        inf
        inf        inf        inf        inf]


## 2. Automated DSGE Reports with `dsge_report()`

The `dsge_report()` function generates a full diagnostic and analytical report of the model.

It automatically includes:
- Metadata & parameter calibrations
- Symbolic equations and LaTeX representations
- Steady-state check and residuals
- Generalized eigenvalues and stability diagnostics
- Decision rules and policy functions
- Impulse response functions and simulation plots


In [4]:
# Generate report for rbc.dyno
report = dsge_report(filename="models/rbc.dyno")

# In Jupyter, calling display() or letting the object return renders the rich HTML/Markdown report
report.display()


/home/pablo/Econforge/dyno-examples/.pixi/envs/default/lib/python3.12/site-packages/dyno/report.py:1285: UserWarning: DynoModel.run() is experimental and its command metadata format may change.
  run_output = model.run(default_pipeline=False)


## 3. JupyterLab Dyno Extension (`jupyterlab-dyno`)

If you are using JupyterLab with the `jupyterlab-dyno` extension installed:
1. **Live Model Preview**: Double-click any `.dyno` or `.mod` file to open a split-view editor with live re-rendering of steady state, checks, and IRFs as you type.
2. **Dyno Options Sidebar**: Use the sidebar panel to configure solver parameters (e.g. perturbation order, horizon, steady state only) on a per-file basis.
3. **Error Highlighting**: Syntax and convergence errors are highlighted directly in the editor on the offending line.

This creates an interactive, reactive development environment for macroeconomic modeling!
